# SingleBehaviorLab demo — segmentation & unsupervised discovery

This notebook walks through the three stages of the unsupervised pipeline:

1. **Segment** — propagate SAM2 masks across a video starting from point prompts.
2. **Register** — crop each frame around the tracked object and extract VideoPrism embeddings.
3. **Cluster** — run UMAP + Leiden on the embeddings and visualise the result.

All outputs are compatible with the GUI, so once the notebook finishes you can open the generated analysis pickle in the Clustering tab (**Load Analysis State**) to inspect individual snippets by clicking UMAP points.

See [`../CLI.md`](../CLI.md) for the equivalent one-shot CLI invocations.

## 1. Configuration

The repository ships a short silent demo video and a matching prompts JSON under `demo/data/segmentation_clustering/`, so this notebook is runnable out of the box. Replace them with your own files if you want to try a different recording.

Flip `FULL_VIDEO = True` to process the entire clip. The default (`False`, 2000 frames) runs end-to-end in a few minutes and is enough to produce a meaningful UMAP.

In [ ]:
from pathlib import Path

DATA_DIR = Path("data/segmentation_clustering").resolve()
VIDEO = DATA_DIR / "Demo_video.mp4"
PROMPTS = DATA_DIR / "sam2_prompts.json"

# Set to True to process the whole video; False (default) runs only the first
# 2000 frames so the full pipeline finishes in a few minutes.
FULL_VIDEO = False
END_FRAME = None if FULL_VIDEO else 2000
SAM2_MODEL = "sam2.1_hiera_large.pt"

STEM = VIDEO.stem + ("_full" if FULL_VIDEO else f"_{END_FRAME}")
MASKS = DATA_DIR / f"{STEM}_masks.h5"
MATRIX = DATA_DIR / f"{STEM}_matrix.npz"
ANALYSIS = DATA_DIR / f"{STEM}_analysis.pkl"
PLOT_PDF = DATA_DIR / f"{STEM}_umap.pdf"

assert VIDEO.is_file(), f"Video file not found: {VIDEO}"
assert PROMPTS.is_file(), f"Prompts file not found: {PROMPTS}"
print(f"Video:       {VIDEO}")
print(f"Prompts:     {PROMPTS}")
print(f"Mode:        {'full video' if FULL_VIDEO else f'first {END_FRAME} frames'}")

## 2. Inspect the prompts JSON

Prompts are stored as a simple list of `(frame_idx, obj_id, x, y, label)` entries. `label = 1` is a foreground click, `label = 0` is a negative click.

In [ ]:
import json

with open(PROMPTS) as f:
    prompt_data = json.load(f)

print(f"Source video: {prompt_data.get('video_path')}")
print(f"Prompt count: {len(prompt_data.get('prompts', []))}\n")
for entry in prompt_data["prompts"][:10]:
    print(entry)

## 3. Segment with SAM2

`run_sam2_segmentation` loads the bundled SAM2 checkpoint, applies the prompts, and propagates masks across the requested frame range. Long videos are processed in 200-frame chunks so the memory-optimised SAM2 fork never has to hold the whole video in GPU memory.

In [ ]:
import singlebehaviorlab as sbl

sbl.segment(
    VIDEO,
    PROMPTS,
    MASKS,
    model_name=SAM2_MODEL,
    start_frame=0,
    end_frame=END_FRAME,
    log_fn=print,
)

print(f"\nMask file written: {MASKS}")

## 4. Extract VideoPrism embeddings

`run_registration` crops each video frame around the SAM2 mask, builds overlapping clips (default: 16 frames, 50% overlap), and runs each clip through the VideoPrism backbone to produce a fixed-length feature vector.

The resulting NPZ files follow the same layout as the GUI Registration tab, so you could also open them via **Clustering tab → Load matrix/metadata** and run clustering interactively.

In [ ]:
params = sbl.RegistrationParams(
    target_fps=25,
    clip_length_frames=16,
    # step_frames defaults to clip_length // 2 (50% overlap)
)

reg_result = sbl.register(
    VIDEO,
    MASKS,
    MATRIX,
    params=params,
    log_fn=print,
)

print(f"\nMatrix:   {reg_result['matrix']}")
print(f"Metadata: {reg_result['metadata']}")

## 5. Cluster and plot UMAP

`run_clustering` normalises the feature matrix, runs UMAP, and computes Leiden or HDBSCAN clusters. It writes a pickle that the Clustering tab can open via **Load Analysis State**, which restores the matrix, embedding, cluster labels, metadata, and the snippet→clip map.

`plot_umap_clusters` renders the UMAP scatter from the in-memory state or from a saved pickle.

In [ ]:
cluster_params = sbl.ClusteringParams(
    method="leiden",
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    leiden_resolution=1.0,
)

sbl.cluster(
    reg_result["matrix"],
    ANALYSIS,
    metadata_path=reg_result["metadata"],
    params=cluster_params,
    log_fn=print,
)

print(f"\nAnalysis state: {ANALYSIS}")

In [ ]:
fig = sbl.plot_umap_clusters(
    ANALYSIS,
    show=True,
    save=PLOT_PDF,
    title=f"UMAP — {STEM}",
    point_size=10,
)
print(f"Saved plot: {PLOT_PDF}")

## 6. Inspect the resulting metadata

The metadata DataFrame carries one row per snippet with the video id, object id, clip range, and — importantly — an absolute `clip_path`. That column is what lets the GUI look up and play a clip when you click its UMAP point.

In [ ]:
import pickle
import pandas as pd

with open(ANALYSIS, "rb") as f:
    state = pickle.load(f)

meta = state.get("metadata")
print("Columns:", list(meta.columns) if meta is not None else "(none)")
print("Clusters:", sorted(set(int(c) for c in state["clusters"])))
print("Snippet→clip map entries:", len(state.get("snippet_to_clip_map", {})))

if meta is not None:
    meta.head()

## Next steps

- Open `ANALYSIS` in the GUI (**Clustering tab → Load Analysis State**) and click UMAP points to preview individual clips.
- Select representative clips from each cluster and feed them back into the **Labeling** tab to bootstrap or refine a supervised behavior classifier — see [`01_behavior_sequencing.ipynb`](01_behavior_sequencing.ipynb).
- Try `method="hdbscan"` or adjust `leiden_resolution` to see how the cluster structure changes.